# When does Parquet reading belong on the GPU?

This notebook is the compact companion to the ramwise.dev study. It reconstructs the published row-group result from a small committed table, then optionally runs a correctness-checked local PyArrow/cuDF comparison. The full 50-million-row orchestration and raw telemetry live in the `gpu-lab` repository.

In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'published_results.csv').exists():
    root = root / 'parquet-gpu-break-even'
rows = list(csv.DictReader((root / 'published_results.csv').open()))
sweep = [row for row in rows if row['study'] == 'row-group-sweep']
len(sweep)

## The row-group tradeoff

All four files were effectively the same size. The layout changed reader overhead, not compression. Smaller groups hurt both readers; one group per file helped cuDF slightly but hurt projected PyArrow reads.

In [ ]:
groups = sorted({int(row['row_group_rows']) for row in sweep})
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for axis, projection in zip(axes, ('all', 'core')):
    selected = {int(row['row_group_rows']): row for row in sweep if row['projection'] == projection}
    axis.plot(groups, [float(selected[g]['pyarrow_seconds']) for g in groups], marker='o', label='PyArrow')
    axis.plot(groups, [float(selected[g]['cudf_seconds']) for g in groups], marker='o', label='cuDF')
    axis.set_xscale('log', base=2)
    axis.set_title(f'{projection} columns')
    axis.set_xlabel('rows per Parquet row group')
    axis.set_ylabel('median elapsed seconds')
    axis.grid(alpha=0.25)
    axis.legend()
fig.suptitle('Wide Zstandard Parquet, 50 million rows')
fig.tight_layout()

The balanced recommendation is **262,144 rows per group**. It is near-optimal for cuDF and stable for PyArrow. One 2.5-million-row group per file is a reasonable cuDF-only full-scan optimization, not a universal default.

## Optional: run the teaching benchmark

Start small. If cuDF is unavailable, this still runs the PyArrow path. Do not treat a laptop-scale result as a reproduction of the 50-million-row study: the size crossover is part of the question.

In [ ]:
import sys
sys.path.insert(0, str(root))
from benchmark_parquet import run_benchmark

local = run_benchmark(
    rows=1_000_000,
    row_group_rows=262_144,
    projection='core',
    trials=3,
    wide=True,
)
local

## What this notebook does not prove

It does not measure cold storage, transfer a CPU dataframe to the GPU, locate the dataset-size break-even point, or test out-of-core execution. Those are different experiments. The published claim is narrower: with warm files, direct Parquet reads, 50 million rows, and this hardware/software stack, cuDF won the isolated target conditions and projection dominated the layout choices.